# Transformers desde (casi) cero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cam2149/icesi-nlp/blob/Entrega3/Sesion3/Entrega3.ipynb)

En este notebook implementaremos un clasificador de noticias en español utilizando transformers. Implementaremos parte de la arquitectura del modelo pieza por pieza para ver como funciona por dentro. Sin embargo, utilizarémos las utilidades de tokenización de huggingface transformers para ayudarnos con esta tarea.

#### Referencias
- Dataset: https://huggingface.co/datasets/MarcOrfilaCarreras/spanish-news
- [Attention is All You Need](http://arxiv.org/abs/1706.03762)
- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246)
- [Tutorial 5: Transformers and Multi-Head Attention](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/05-transformers-and-MH-attention.html)

In [ ]:
!pip install optuna

In [11]:
# Ahora vamos a crear nuestro implementación de clasificador de reseñas de Amazon en español
# Basándonos en la estructura del archivo adjunto pero adaptándolo para el dataset de Amazon

# Primero instalamos las dependencias necesarias
import sys
import subprocess

def install_packages():
    packages = [
        'torch',
        'transformers',
        'datasets',
        'tokenizers',
        'scikit-learn',
        'matplotlib',
        'seaborn',
        'pandas',
        'numpy',
        'tqdm',
        'os'
    ]

    for package in packages:
        try:
            __import__(package)
            print(f"✓ {package} ya está instalado")
        except ImportError:
            print(f"Instalando {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# install_packages()

# Importaciones principales
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import os
import subprocess
import sys
from typing import Optional, Tuple
from torch.utils.data import Dataset, DataLoader, random_split
import warnings
warnings.filterwarnings('ignore')

print("Dependencias cargadas correctamente")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"Número de GPUs: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo seleccionado: {device}")


Dependencias cargadas correctamente
PyTorch version: 2.8.0+cu126
CUDA disponible: True
Número de GPUs: 1
Dispositivo seleccionado: cuda


In [12]:
# Instalamos solo torch primero
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "--quiet", "--no-cache-dir"])
    print("✓ PyTorch instalado")
except:
    print("Usando versión del sistema")

✓ PyTorch instalado


In [13]:
warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

In [14]:
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets 'transformers[torch]' sentence-transformers optuna

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

### Cargando el dataset
Este es un dataset pequeño de articulos de noticias en idioma español con sus respectivas categorías. El dataset está disponible en el HuggingFace Hub y puede ser fácilmente descargado con la librería.

In [15]:
# 1. Cargar el dataset
def load_spanish_news_dataset():
    """Carga el dataset de noticias en español"""
    dataset_train = load_dataset("MarcOrfilaCarreras/spanish-news", split='train')
    dataset_test = load_dataset("MarcOrfilaCarreras/spanish-news", split='test')
    dataset_test = load_dataset("MarcOrfilaCarreras/spanish-news", split='test')
    return dataset

Observemos uno de sus registros...

In [ ]:
# Cargar dataset
print("Cargando dataset...")
dataset = load_spanish_news_dataset()
dataset = dataset.shuffle(seed=42)

# Acceder a un ejemplo de la división de entrenamiento
print("Observando un ejemplo del dataset:")
example = dataset[0]
print(example)

Cargando dataset...
Observando un ejemplo del dataset:
{'language': 'es', 'category': 'economy', 'newspaper': 'investing', 'hash': 'b43d231ec84d39df25d9252fdaf3423b1b55845a', 'text': 'Madrid, 20 feb (.).- La Bolsa española ha abierto, un día más, con leves movimientos, y en los primeros compases de este martes baja el 0,04 %, en una jornada en la que el mercado volverá a contar con la referencia de Wall Street, que ayer cerró por festivo.El IBEX 35, el principal selectivo español, cotiza en los 9.939,3 puntos en la apertura, tras dejarse ese mínimo 0,04 %. Las pérdidas acumuladas en el año alcanzan el 1,61 %.En una nueva jornada en la que los inversores no contarán con referencias macroeconómicas relevantes, el mercado estará pendiente de los resultados, mientras en España Enagás (BME:ENAG) ha anunciado un beneficio de 342,5 millones en 2023, un 8,8 % menos. Sus acciones suben el 3,44 %.   (foto)(vídeo)'}


Para los efectos de esta tarea, nos servirán el texto y la categoría naturalmente.

A manera general, observemos que tan largos o cortos tienden a ser los textos.

In [ ]:
text_lengths = [len(row['text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

Texto más corto: 501
Texto más largo: 204324
Longitud promedio: 4218.154509803921


Estos valores son la cantidad de *caractéres* que tiene las secuencias. Una decisión ingenua pero útil en este momento podría ser ajustar la longitud de las secuencias que vamos a usar para el entrenamiento a unos 2000 tokens. Esto podría ser suficiente para capturar una porción significativa de los textos.

## Definiendo el Tokenizer

Ahora, vamos a definir el tokenizer para nuestra tarea. Para ahorrarnos tiempo, vamos a entrenar uno basado en gpt2, pero ajustandolo a nuestro dataset. Para ello, debemos seleccionar una muestra representativa de nuestro dataset, como no es muy grande, casi que podemos usarlo todo. Luego, debemos definir el tamaño del vocabulario, es decir, cuantos tokens únicos queremos soportar en nuestro tokenizador. Para que un modelo de lenguaje funcione moderadamente bien para una tarea de clasificación, considerando el tamaño de nuestro corpus, deberíamos definir unos 50 mil tokens.

In [ ]:
# 3. Implementación de un bloque Transformer
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerBlock, self).__init__()

        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention con conexión residual
        attn_output, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Feed-forward con conexión residual
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x


In [ ]:
# 4. Embeddings posicionales
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length=512):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_length, d_model)
        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
# 5. Clasificador completo de noticias
class SpanishNewsClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=6,
                 num_classes=12, max_length=512, dropout=0.1):
        super(SpanishNewsClassifier, self).__init__()

        self.d_model = d_model
        self.max_length = max_length

        # Embeddings de tokens
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_length)

        # Capas transformer
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_model * 4, dropout)
            for _ in range(num_layers)
        ])

        # Capa de clasificación
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask=None):
        # Embeddings
        x = self.token_embedding(input_ids) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)
        x = self.dropout(x)

        # Pasar por las capas transformer
        for transformer in self.transformer_blocks:
            x = transformer(x, attention_mask)

        # Pooling: usar el token [CLS] o promedio
        if attention_mask is not None:
            # Promedio ponderado por la máscara de atención
            mask_expanded = attention_mask.unsqueeze(-1).expand(x.size()).float()
            sum_embeddings = torch.sum(x * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            pooled = sum_embeddings / sum_mask
        else:
            # Promedio simple
            pooled = x.mean(dim=1)

        # Clasificación
        logits = self.classifier(pooled)
        return logits


In [ ]:
# 6. Dataset personalizado
class SpanishNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenización usando HuggingFace
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [9]:
# 7. Función principal de entrenamiento
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset

def main():
    # Configuración
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Cargar tokenizer de HuggingFace (usamos BETO, un modelo español)
    tokenizer = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')

    # Cargar dataset
    print("Cargando dataset...")
    dataset = load_spanish_news_dataset()

    # Preparar datos
    # Access data directly from the dataset object
    texts = dataset['text'][:1000]  # Muestra pequeña para demo
    categories = dataset['category'][:1000]


    # Convertir categorías a números
    unique_categories = list(set(categories))
    category_to_id = {cat: idx for idx, cat in enumerate(unique_categories)}
    labels = [category_to_id[cat] for cat in categories]

    print(f"Categorías encontradas: {unique_categories}")
    print(f"Número de clases: {len(unique_categories)}")

    # Crear dataset
    news_dataset = SpanishNewsDataset(texts, labels, tokenizer)
    dataloader = DataLoader(news_dataset, batch_size=8, shuffle=True)

    # Crear modelo
    model = SpanishNewsClassifier(
        vocab_size=tokenizer.vocab_size,
        d_model=256,
        num_heads=8,
        num_layers=4,
        num_classes=len(unique_categories)
    ).to(device)

    # Optimizador y función de pérdida
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Entrenamiento básico
    model.train()
    for epoch in range(2):  # Solo 2 épocas para demo
        total_loss = 0
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()

            # Forward pass
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')

        avg_loss = total_loss / len(dataloader)
        print(f'Epoch {epoch+1} completada - Loss promedio: {avg_loss:.4f}')

if __name__ == "__main__":
    main()

KeyboardInterrupt: 